
# Financial Fraud Detection Analytics & Machine Learning
## End-to-End Fraud Analysis, Risk Intelligence & Predictive Modeling

This notebook provides a professional-grade fraud analytics workflow using machine learning and advanced exploratory analysis.

## Included Sections
- Full Exploratory Data Analysis (EDA)
- Data Cleaning & Preprocessing
- Missing Value Analysis
- Outlier Detection
- Feature Engineering
- Fraud Pattern Analysis
- Risk Insights
- Correlation Analysis
- Transaction Behavior Analysis
- Advanced Visualizations
- Predictive Fraud Detection Models
- Feature Importance Analysis
- Business Recommendations
- Production-Quality Code & Documentation

---

## Dataset Overview
- Rows: **7,000**
- Columns: **13**



In [ ]:

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")


In [ ]:

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(r"/mnt/data/fraud.csv")

print("Dataset Shape:", df.shape)

df.head()


## Dataset Inspection

In [ ]:

df.info()


In [ ]:

df.describe(include='all').T


## Missing Value Analysis

In [ ]:

missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values,
    'Missing Percentage': (missing.values / len(df)) * 100
})

missing_df.head(20)


In [ ]:

plt.figure(figsize=(14,6))

sns.barplot(
    x=missing_df['Column'][:20],
    y=missing_df['Missing Percentage'][:20]
)

plt.xticks(rotation=90)

plt.title("Top Missing Value Percentages")

plt.show()


## Data Cleaning & Preprocessing

In [ ]:

# Remove duplicates

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

df = df.drop_duplicates()

print("Shape After Deduplication:", df.shape)


In [ ]:

# Automatic numeric conversion

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print("Numeric conversion completed.")


## Exploratory Data Analysis

In [ ]:

# Numerical Features

numeric_cols = df.select_dtypes(include='number').columns.tolist()

print(numeric_cols)


In [ ]:

# Distribution Analysis

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:6]:

    plt.figure(figsize=(10,5))

    sns.histplot(df[col].dropna(), kde=True)

    plt.title(f"Distribution of {col}")

    plt.show()


## Fraud Class Distribution

In [ ]:

# Detect potential fraud target column automatically

possible_targets = [
    c for c in df.columns
    if 'fraud' in c.lower()
    or 'target' in c.lower()
    or 'class' in c.lower()
]

print("Potential Target Columns:", possible_targets)

if len(possible_targets) > 0:

    target_col = possible_targets[0]

    plt.figure(figsize=(8,5))

    sns.countplot(x=df[target_col])

    plt.title(f"Fraud Distribution - {target_col}")

    plt.show()

    print(df[target_col].value_counts())


## Correlation Analysis

In [ ]:

numeric_df = df.select_dtypes(include='number')

corr = numeric_df.corr()

plt.figure(figsize=(16,10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    annot=False
)

plt.title("Correlation Matrix")

plt.show()


## Outlier Detection

In [ ]:

# Boxplots for Numerical Features

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:5]:

    plt.figure(figsize=(12,4))

    sns.boxplot(x=df[col])

    plt.title(f"Outlier Detection - {col}")

    plt.show()


In [ ]:

# IQR-Based Outlier Detection Example

if len(numeric_cols) > 0:

    col = numeric_cols[0]

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"Outliers in {col}: {len(outliers)}")


## Fraud Pattern & Risk Analysis

In [ ]:

# High-Risk Transaction Exploration

numeric_cols = df.select_dtypes(include='number').columns.tolist()

if len(numeric_cols) > 1:

    plt.figure(figsize=(10,6))

    sns.scatterplot(
        x=df[numeric_cols[0]],
        y=df[numeric_cols[1]]
    )

    plt.title("Transaction Pattern Analysis")

    plt.show()


## Feature Engineering

In [ ]:

# Feature Engineering

# Missing value count feature
df['missing_feature_count'] = df.isnull().sum(axis=1)

# Transaction intensity score example
numeric_cols = df.select_dtypes(include='number').columns.tolist()

if len(numeric_cols) >= 2:

    df['transaction_intensity'] = (
        df[numeric_cols[0]].fillna(0) *
        df[numeric_cols[1]].fillna(0)
    )

df.head()


## Advanced Visualizations

In [ ]:

# Pairplot

important_numeric = df.select_dtypes(include='number').columns.tolist()[:5]

if len(important_numeric) > 1:

    sns.pairplot(
        df[important_numeric].dropna()
    )

    plt.show()


## Predictive Machine Learning Model

In [ ]:

# =========================
# FRAUD DETECTION MODEL
# =========================

# Identify target column automatically

possible_targets = [
    c for c in df.columns
    if 'fraud' in c.lower()
    or 'target' in c.lower()
    or 'class' in c.lower()
]

if len(possible_targets) > 0:

    target = possible_targets[0]

else:

    target = df.select_dtypes(include='number').columns[-1]

print("Selected Target:", target)

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

categorical_features = X.select_dtypes(include='object').columns.tolist()

numeric_features = [
    c for c in X.columns
    if c not in categorical_features
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Remove rows with missing target
valid_idx = y.notnull()

X = X[valid_idx]
y = y[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

# Metrics
acc = accuracy_score(y_test, preds)

print("Accuracy:", round(acc, 4))

print("\nClassification Report:")
print(classification_report(y_test, preds))

try:
    probs = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, probs)
    print("ROC-AUC:", round(auc, 4))
except:
    print("ROC-AUC unavailable.")


## Confusion Matrix

In [ ]:

# Confusion Matrix

cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


## Feature Importance

In [ ]:

# Feature Importance

rf_model = model.named_steps['model']

encoded_cat = model.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .named_steps['encoder']\
    .get_feature_names_out(categorical_features)

all_features = numeric_features + list(encoded_cat)

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

top_features = importance_df.head(20)

plt.figure(figsize=(12,8))

sns.barplot(
    x='Importance',
    y='Feature',
    data=top_features
)

plt.title("Top 20 Important Features")

plt.show()

top_features



# Business Insights & Recommendations

## Key Fraud Insights
- Fraudulent transactions often exhibit abnormal transaction patterns.
- Outlier behavior can strongly correlate with fraud risk.
- Feature engineering significantly improves fraud detection performance.
- Class imbalance is a critical challenge in fraud analytics.

## Recommendations
### For Financial Institutions
- Deploy real-time fraud scoring systems.
- Use anomaly detection pipelines.
- Continuously retrain fraud models.

### For Risk Teams
- Monitor high-risk transaction clusters.
- Track abnormal behavior deviations.
- Combine ML predictions with rule-based systems.

### For Data Science Teams
- Implement SMOTE or imbalance handling.
- Explore XGBoost/LightGBM models.
- Use deep learning for sequence-based fraud detection.

## Future Improvements
- Real-time streaming analytics
- Graph-based fraud detection
- Behavioral biometrics
- Ensemble fraud models
- Explainable AI (SHAP/LIME)



# Conclusion

This notebook demonstrates a complete professional fraud analytics workflow.

The project includes:
- Data preprocessing
- Advanced EDA
- Fraud pattern analysis
- Feature engineering
- Machine learning classification
- Feature importance analysis
- Risk intelligence insights

This framework can scale into:
- Enterprise fraud detection systems
- Real-time risk scoring engines
- Banking security analytics platforms
- AI-driven financial monitoring tools
